# Serve the official N-ATLaS for the Sovereign Gateway

Runs **NCAIR1/N-ATLaS** (Llama-3 8B, NCAIR/Awarri) on a free Colab GPU in 4-bit with Hugging Face transformers, behind an OpenAI-compatible API, and exposes it through a temporary Cloudflare tunnel for the N-ATLAS Sovereign Gateway.

**Before you start**
1. *Runtime > Change runtime type > T4 GPU*.
2. Accept the model terms at https://huggingface.co/NCAIR1/N-ATLaS (access is approved automatically, instantly).
3. Create a Hugging Face **read** token and add it in Colab **Secrets** (key icon on the left) as `HF_TOKEN`, with notebook access on.

Then *Runtime > Run all*. The last-but-one cell prints the URL and key for the gateway. Keep the tab open while you use it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Model server dependencies, the tunnel binary, and the server script from the project repo.
!pip -q install -U transformers accelerate bitsandbytes fastapi uvicorn
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!wget -q -O natlas_server.py https://raw.githubusercontent.com/Em-driod/natlas-sovereign-gateway/main/deploy/colab/natlas_server.py
!ls -la natlas_server.py && cloudflared --version

In [ ]:
import os, secrets
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()
# Key the gateway must send, so only your gateway can use this endpoint.
API_KEY = "natlas-" + secrets.token_urlsafe(18)
os.environ["UPSTREAM_KEY"] = API_KEY
print("Token loaded, upstream key generated.")

In [ ]:
# Start the N-ATLaS server in the background (logs go to server.log).
import subprocess
server = subprocess.Popen(["python", "natlas_server.py"], env=os.environ.copy(), stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
print("Server starting. Downloading and loading the model takes about 5 to 15 minutes on first run.")

In [ ]:
# Wait until the model is loaded and answering.
import time, requests
for i in range(150):
    try:
        if requests.get("http://localhost:8000/health", timeout=3).ok:
            print("N-ATLaS is up after", i * 10, "seconds"); break
    except Exception:
        pass
    if server.poll() is not None:
        print("Server stopped. Last log lines:"); print(open("server.log").read()[-4000:]); break
    if i % 6 == 0:
        last = open("server.log").read().strip().splitlines()[-1:]
        print(f"{i * 10}s:", last[0][:150] if last else "starting...")
    time.sleep(10)
else:
    print(open("server.log").read()[-4000:])

In [ ]:
# Sanity check in four Nigerian languages, straight to the model.
import requests
H = {"Authorization": f"Bearer {API_KEY}"}
for q in ["Translate 'Good morning, how can I help you today?' to Yoruba.",
          "Translate 'Good morning, how can I help you today?' to Hausa.",
          "Translate 'Good morning, how can I help you today?' to Igbo.",
          "Explain wetin BVN be for Pidgin, short."]:
    r = requests.post("http://localhost:8000/v1/chat/completions", headers=H, timeout=180, json={
        "model": "n-atlas-8b", "temperature": 0.2, "max_tokens": 80,
        "messages": [{"role": "user", "content": q}]}).json()
    print("Q:", q)
    print("A:", r["choices"][0]["message"]["content"].strip())
    print()

In [ ]:
# Public HTTPS tunnel (temporary URL, valid while this notebook runs).
import re, subprocess
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+[.]trycloudflare[.]com", line)
    if m:
        url = m.group(0); break
print()
print("================ GIVE THESE TO THE GATEWAY ================")
print("UPSTREAM_URL     =", f"{url}/v1")
print("UPSTREAM_API_KEY =", API_KEY)
print("============================================================")

In [ ]:
# Keep this cell running so the tunnel stays open while you demo or record.
import time
while True:
    time.sleep(60)